# Concatenar UTCI diario (ERA5-HEAT) → un solo NetCDF de México 2023

Lee los archivos diarios `ECMWF_utci_YYYYMMDD_v1.1_con.nc`,
recorta al dominio de México y concatena todo el año en un solo `.nc`.

Requiere: `xarray`, `netcdf4`, `dask` (opcional pero recomendado).

In [ ]:
from pathlib import Path
import glob

import numpy as np
import xarray as xr

# Ajusta esta ruta a donde estén montados los archivos en el servidor
DATA_DIR = Path("/ruta/al/NAS-eolica/3_data/ERA5/confort")
YEAR = 2023

# Dominio de México (con un poco de margen)
LAT_MIN, LAT_MAX = 14.0, 33.5
LON_MIN, LON_MAX = -118.5, -86.0  # en convención -180..180

OUT_FILE = Path(f"UTCI_Mexico_{YEAR}.nc")

In [ ]:
files = sorted(glob.glob(str(DATA_DIR / f"ECMWF_utci_{YEAR}*_v1.1_con.nc")))
print(f"Archivos encontrados: {len(files)}")  # esperamos 365
print("Primero:", files[0])
print("Último: ", files[-1])

In [ ]:
ds0 = xr.open_dataset(files[0])
print(ds0)
# Anota: nombre de la variable (usualmente 'utci'), unidades (K),
# nombres de coords ('lat'/'latitude', 'lon'/'longitude') y rango de lon (0..360 o -180..180)

In [ ]:
def preprocess_mexico(ds: xr.Dataset) -> xr.Dataset:
    # Normalizar nombres de coordenadas
    ren = {}
    if "latitude" in ds.coords:
        ren["latitude"] = "lat"
    if "longitude" in ds.coords:
        ren["longitude"] = "lon"
    if ren:
        ds = ds.rename(ren)

    # Pasar longitudes 0..360 → -180..180 si hace falta
    if float(ds.lon.max()) > 180:
        ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby("lon")

    # Latitud puede venir descendente (típico de ERA5)
    if ds.lat[0] > ds.lat[-1]:
        lat_slice = slice(LAT_MAX, LAT_MIN)
    else:
        lat_slice = slice(LAT_MIN, LAT_MAX)

    return ds.sel(lat=lat_slice, lon=slice(LON_MIN, LON_MAX))

In [ ]:
test = preprocess_mexico(ds0)
print(test)
print("lat:", float(test.lat.min()), "→", float(test.lat.max()))
print("lon:", float(test.lon.min()), "→", float(test.lon.max()))
ds0.close()

In [ ]:
# Bucle secuencial: cada día recortado a México pesa ~1 MB en memoria,
# así el kernel nunca carga los campos globales de golpe.
partes = []
for i, f in enumerate(files):
    with xr.open_dataset(f) as d:
        partes.append(preprocess_mexico(d).load())
    if (i + 1) % 30 == 0 or i == len(files) - 1:
        print(f"{i + 1}/{len(files)}  {Path(f).name}")

In [ ]:
ds = xr.concat(partes, dim="time")
ds = ds.sortby("time")
del partes
print(ds)
print("Pasos de tiempo:", ds.sizes["time"])  # esperamos 8760 (365 días × 24 h)

In [ ]:
# Descomenta si prefieres guardar en °C
# ds["utci"] = ds["utci"] - 273.15
# ds["utci"].attrs["units"] = "degC"

In [ ]:
encoding = {
    var: {"zlib": True, "complevel": 4, "dtype": "float32"}
    for var in ds.data_vars
}
ds.to_netcdf(OUT_FILE, encoding=encoding)
print("Guardado:", OUT_FILE.resolve())

In [ ]:
check = xr.open_dataset(OUT_FILE)
print(check)
print("time:", str(check.time.values[0]), "→", str(check.time.values[-1]))
print("¿huecos en el tiempo?",
      not bool(np.all(np.diff(check.time.values).astype("timedelta64[h]")
                      == np.timedelta64(1, "h"))))
check.close()